# 🏥 TopoNet Ablation Study & Replication (MICCAI 2025)
### Laparoscopic Liver Landmark Detection on L3D Dataset
**Automated End-to-End Kaggle GPU Notebook**

---

### Key Goals & Design
1. **Paper Replication (Table 2 & Table 1):**
   - Replicate the 6 ablation modes on the Validation split (122 frames).
   - Evaluate the full model on the Test split (109 frames) to benchmark against paper SOTA.
2. **Patient 32 4K Canvas Bug Fix:**
   - Automatically reads `imageHeight` and `imageWidth` directly from Label JSONs to avoid coordinate truncation on 4K images.
3. **Patient 40 Failure Analysis:**
   - Automatically isolates and logs metrics for Patient 40, and renders 4-panel visual diagnostic images (`RGB`, `Ground Truth`, `TopoNet Pred`, `Error Map`).
4. **Automated Result Packaging:**
   - Automatically packages all checkpoints, metrics CSVs, JSON summaries, and visual diagnostic panels into `/kaggle/working/EXPERIMENT_1_RESULTS.zip` for instant one-click download.


## Step 1: Environment & GPU Acceleration Check
Verify CUDA availability, GPU device specifications, and workspace folders.


In [ ]:
import os
import sys
import glob
import subprocess
import torch

print("=" * 70)
print("🚀 SYSTEM & GPU DIAGNOSTICS")
print("=" * 70)
print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Active GPU     : {gpu_name}")
    print(f"Total VRAM     : {total_mem:.2f} GB")
else:
    print("⚠️ WARNING: Running on CPU! Make sure GPU Accelerator is turned ON in Kaggle sidebar!")
print("=" * 70)

# Create structured workspace directories
os.makedirs('/kaggle/working/repos', exist_ok=True)
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/results', exist_ok=True)
os.makedirs('/kaggle/working/experiments/EXPERIMENT_1/models', exist_ok=True)
os.makedirs('/kaggle/working/experiments/EXPERIMENT_1/utils', exist_ok=True)
os.makedirs('/kaggle/working/experiments/EXPERIMENT_1/scripts', exist_ok=True)
print("✅ Workspace directories initialized under /kaggle/working/")


## Step 2: Install Dependencies & Compile Betti Matching 3D
Install surface distance evaluation packages and compile the C++ persistent homology library (`Betti-Matching-3D`).


In [ ]:
# 1. Install evaluation & vision libraries + pybind11
!pip install -q surface-distance medpy einops timm pybind11

# 2. Build Betti Matching 3D C++ pybind module
print("📦 Compiling Betti Matching 3D C++ module...")
!mkdir -p /kaggle/working/betti_match/Betti_Matching
if not os.path.exists('/kaggle/working/betti_match/Betti_Matching/CMakeLists.txt'):
    !git clone --depth 1 https://github.com/nstucki/Betti-Matching-3D.git /kaggle/working/betti_match/Betti_Matching

import pybind11
pybind_cmake_dir = pybind11.get_cmake_dir()

%cd /kaggle/working/betti_match/Betti_Matching
!mkdir -p betti_build
%cd betti_build
!cmake .. -Dpybind11_DIR="{pybind_cmake_dir}" -DCMAKE_BUILD_TYPE=Release
!make -j4
%cd /kaggle/working

# Touch __init__.py files for clean Python module imports
!touch /kaggle/working/betti_match/__init__.py
!touch /kaggle/working/betti_match/Betti_Matching/__init__.py
!touch /kaggle/working/betti_match/Betti_Matching/betti_build/__init__.py

# Add to sys.path
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

# Verify Betti import
try:
    from betti_match.Betti_Matching.betti_build import betti_matching
    print("✅ Betti Matching C++ library compiled and imported successfully!")
except Exception as e:
    print(f"⚠️ Notice: Betti import check: {e}")


## Step 3: Clone Official TopoNet Codebase
Clone the official TopoNet repository into `/kaggle/working/repos/TopoNet` for reference modules (ResNet, DSCNet, PPM Decoder).


In [ ]:
if not os.path.exists('/kaggle/working/repos/TopoNet'):
    print("📥 Cloning cuiruize/TopoNet...")
    !git clone --depth 1 https://github.com/cuiruize/TopoNet.git /kaggle/working/repos/TopoNet
    print("✅ TopoNet reference repo cloned.")
else:
    print("✅ TopoNet repo already exists.")

if '/kaggle/working/repos/TopoNet' not in sys.path:
    sys.path.append('/kaggle/working/repos/TopoNet')


## Step 4: Download Depth Anything V2 ViT-B Weights
Download the official pretrained Depth Anything V2 ViT-B weights (`depth_anything_v2_vitb.pth`, ~390 MB).


In [ ]:
depth_ckpt = '/kaggle/working/checkpoints/depth_anything_v2_vitb.pth'
if not os.path.exists(depth_ckpt) or os.path.getsize(depth_ckpt) < 100_000_000:
    print("📥 Downloading Depth Anything V2 ViT-B weights from HuggingFace...")
    !wget -q --show-progress -O {depth_ckpt} "https://huggingface.co/depth-anything/Depth-Anything-V2-Base/resolve/main/depth_anything_v2_vitb.pth"
    print(f"✅ Depth weights saved to {depth_ckpt} ({os.path.getsize(depth_ckpt) / (1024**2):.1f} MB)")
else:
    print(f"✅ Depth weights already cached at {depth_ckpt}")


## Step 5: Automatic Dataset Path Discovery & Verification
Scan `/kaggle/input` to locate `Train`, `Val`, and `Test` directories and verify Patient 32 4K canvas handling.


In [ ]:
def find_dataset_split(split_keyword):
    candidates = []
    target = split_keyword.lower()
    for root, dirs, files in os.walk('/kaggle/input'):
        has_imgs = 'images' in dirs or any(f.lower().endswith(('.jpg', '.png')) for f in files)
        if not has_imgs:
            continue
        
        root_lower = root.lower()
        if target in root_lower:
            score = 0
            if 'images' in dirs:
                score += 10
            if 'labels' in dirs:
                score += 10
            if any('patient' in f.lower() for f in files):
                score += 5
            if os.path.basename(root).lower() == target:
                score += 20
            candidates.append((score, root))

    if candidates:
        candidates.sort(key=lambda x: (x[0], len(x[1])), reverse=True)
        return candidates[0][1]
    return None

print("🔍 Inspecting /kaggle/input structure:")
for root, dirs, files in os.walk('/kaggle/input'):
    if dirs or any(f.endswith('.jpg') for f in files):
        print(f"   Folder: {root} -> subdirs: {dirs[:4]}")

train_dir = find_dataset_split('train')
val_dir = find_dataset_split('val')
test_dir = find_dataset_split('test')

print("=" * 70)
print("📂 DISCOVERED DATASET SPLIT LOCATIONS")
print("=" * 70)
print(f"Train Dir: {train_dir} (Exists: {os.path.exists(train_dir) if train_dir else False})")
print(f"Val Dir  : {val_dir} (Exists: {os.path.exists(val_dir) if val_dir else False})")
print(f"Test Dir : {test_dir} (Exists: {os.path.exists(test_dir) if test_dir else False})")
print("=" * 70)

if not train_dir or not val_dir or not test_dir:
    raise FileNotFoundError(
        f"❌ Could not automatically locate all 3 dataset splits in /kaggle/input!\n"
        f"   Found: Train={train_dir}, Val={val_dir}, Test={test_dir}\n"
        f"   Please confirm that l3d-train, l3d-val, and l3d-test are attached in the Kaggle right-hand panel."
    )


## Step 6: Deploy Experiment Modules
Deploy decoupled Dataset with Patient 32 4K fix, Evaluation Metrics with ASSD, Unified TopoNet Ablation Model, and Training Runner.


In [ ]:
import os
import base64

# 1. Ensure directories and __init__.py files exist
for d in [
    '/kaggle/working/experiments/EXPERIMENT_1/models',
    '/kaggle/working/experiments/EXPERIMENT_1/utils',
    '/kaggle/working/experiments/EXPERIMENT_1/scripts',
]:
    os.makedirs(d, exist_ok=True)

for p in [
    '/kaggle/working/experiments/__init__.py',
    '/kaggle/working/experiments/EXPERIMENT_1/__init__.py',
    '/kaggle/working/experiments/EXPERIMENT_1/models/__init__.py',
    '/kaggle/working/experiments/EXPERIMENT_1/utils/__init__.py',
    '/kaggle/working/experiments/EXPERIMENT_1/scripts/__init__.py',
]:
    with open(p, 'w') as f:
        pass

# 2. Deploy dataset.py
with open('/kaggle/working/experiments/EXPERIMENT_1/utils/dataset.py', 'wb') as f:
    f.write(base64.b64decode('aW1wb3J0IG9zCmltcG9ydCBnbG9iCmltcG9ydCBqc29uCmltcG9ydCBjdjIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQKZnJvbSB0b3JjaHZpc2lvbiBpbXBvcnQgdHJhbnNmb3JtcyBhcyBUCgoKY2xhc3MgVG9wb05ldERhdGFzZXQoRGF0YXNldCk6CiAgICAiIiIKICAgIFJvYnVzdCBMM0QgRGF0YXNldCBSZWFkZXIgZm9yIFRvcG9OZXQgd2l0aCBkeW5hbWljIGNhbnZhcyBzaXppbmcuCiAgICBGaXhlcyB0aGUgUGF0aWVudCAzMiA0SyBjYW52YXMgdHJ1bmNhdGlvbiBidWcgYnkgcmVhZGluZyBpbWFnZUhlaWdodC9pbWFnZVdpZHRoCiAgICBkaXJlY3RseSBmcm9tIHRoZSBMYWJlbCBKU09OLgogICAgIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgZGF0YV9kaXIsIHRyYW5zZm9ybT1Ob25lLCBtb2RlPSd0cmFpbicpOgogICAgICAgIHNlbGYuZGF0YV9kaXIgPSBkYXRhX2RpcgogICAgICAgIHNlbGYubW9kZSA9IG1vZGUKICAgICAgICAKICAgICAgICAjIFJlc29sdmUgaW1hZ2VzIGRpcmVjdG9yeSBmbGV4aWJseSAoc3VwcG9ydHMgYm90aCBuZXN0ZWQgJ2ltYWdlcy8nIGFuZCBmbGF0IGZvbGRlcnMpCiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMob3MucGF0aC5qb2luKGRhdGFfZGlyLCAnaW1hZ2VzJykpOgogICAgICAgICAgICBzZWxmLmltYWdlX3BhdGhzID0gc29ydGVkKGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4oZGF0YV9kaXIsICdpbWFnZXMnLCAnKi5bakpdW3BQXVtnR10nKSkgKyAKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4oZGF0YV9kaXIsICdpbWFnZXMnLCAnKi5bcFBdW25OXVtnR10nKSkpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5pbWFnZV9wYXRocyA9IHNvcnRlZChnbG9iLmdsb2Iob3MucGF0aC5qb2luKGRhdGFfZGlyLCAnKi5bakpdW3BQXVtnR10nKSkgKyAKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4oZGF0YV9kaXIsICcqLltwUF1bbk5dW2dHXScpKSkKICAgICAgICAgICAgCiAgICAgICAgaWYgbGVuKHNlbGYuaW1hZ2VfcGF0aHMpID09IDA6CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTm8gaW1hZ2VzIGZvdW5kIGluIGRhdGFzZXQgZGlyZWN0b3J5OiB7ZGF0YV9kaXJ9IikKCiAgICAgICAgc2VsZi50cmFuc2Zvcm0gPSB0cmFuc2Zvcm0gaWYgdHJhbnNmb3JtIGVsc2Ugc2VsZi5fZGVmYXVsdF90cmFuc2Zvcm0KCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX2RlZmF1bHRfdHJhbnNmb3JtKGltYWdlLCBtYXNrLCBkZXB0aCk6CiAgICAgICAgdG9fdGVuc29yID0gVC5Ub1RlbnNvcigpCiAgICAgICAgcmV0dXJuIHRvX3RlbnNvcihpbWFnZSksIHRvX3RlbnNvcihtYXNrKSwgdG9fdGVuc29yKGRlcHRoKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiBsZW4oc2VsZi5pbWFnZV9wYXRocykKCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaWR4KToKICAgICAgICBpbWdfcGF0aCA9IHNlbGYuaW1hZ2VfcGF0aHNbaWR4XQogICAgICAgIGltYWdlID0gc2VsZi5sb2FkX2ltYWdlKGltZ19wYXRoKQogICAgICAgIG1hc2sgPSBzZWxmLmxvYWRfbWFzayhpbWdfcGF0aCkKICAgICAgICBkZXB0aCA9IG5wLnplcm9zKCgxMDI0LCAxMDI0KSwgZHR5cGU9bnAudWludDgpICAjIFRvcG9OZXQgaW5mZXJzIGRlcHRoIGR5bmFtaWNhbGx5IHZpYSBkZXB0aF9lbmNvZGVyCgogICAgICAgICMgVHJhbnNmb3JtIGV4cGVjdHMgbWFzayBpbiBzaGFwZSAoSCwgVywgQykKICAgICAgICBpbWFnZV90LCBtYXNrX3QsIGRlcHRoX3QgPSBzZWxmLnRyYW5zZm9ybShpbWFnZSwgbWFzay50cmFuc3Bvc2UoMSwgMiwgMCksIGRlcHRoKQoKICAgICAgICByZXR1cm4gaW1hZ2VfdCwgZGVwdGhfdCwgbWFza190LCBvcy5wYXRoLmJhc2VuYW1lKGltZ19wYXRoKQoKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBsb2FkX2ltYWdlKHBhdGgpOgogICAgICAgIGltZyA9IGN2Mi5pbXJlYWQoc3RyKHBhdGgpKQogICAgICAgIGlmIGltZyBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkNvdWxkIG5vdCBsb2FkIGltYWdlIGF0IHtwYXRofSIpCiAgICAgICAgaW1nID0gY3YyLnJlc2l6ZShpbWcsICgxMDI0LCAxMDI0KSkKICAgICAgICByZXR1cm4gY3YyLmN2dENvbG9yKGltZywgY3YyLkNPTE9SX0JHUjJSR0IpCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGxvYWRfbWFzayhpbWdfcGF0aCk6CiAgICAgICAgIiIiCiAgICAgICAgRHluYW1pY2FsbHkgcmVuZGVycyBncm91bmQgdHJ1dGggbWFzayBmcm9tIHRoZSBjb3JyZXNwb25kaW5nIEpTT04gbGFiZWwgZmlsZS4KICAgICAgICBVc2VzIGV4YWN0IGltYWdlIGRpbWVuc2lvbnMgZnJvbSBKU09OIG1ldGFkYXRhIHRvIHByZXZlbnQgY29vcmRpbmF0ZSBjbGlwcGluZy4KICAgICAgICAiIiIKICAgICAgICAjIFJlc29sdmUgbGFiZWwgcGF0aAogICAgICAgIGlmICdpbWFnZXMnIGluIHN0cihpbWdfcGF0aCk6CiAgICAgICAgICAgIGpzb25fcGF0aCA9IHN0cihpbWdfcGF0aCkucmVwbGFjZSgnaW1hZ2VzJywgJ2xhYmVscycpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcGFyZW50ID0gb3MucGF0aC5kaXJuYW1lKGltZ19wYXRoKQogICAgICAgICAgICBqc29uX3BhdGggPSBvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKHBhcmVudCksICdsYWJlbHMnLCBvcy5wYXRoLmJhc2VuYW1lKGltZ19wYXRoKSkKICAgICAgICAgICAgCiAgICAgICAganNvbl9wYXRoID0gb3MucGF0aC5zcGxpdGV4dChqc29uX3BhdGgpWzBdICsgJy5qc29uJwoKICAgICAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoanNvbl9wYXRoKToKICAgICAgICAgICAgIyBGYWxsYmFjazogY2hlY2sgYWxvbmdzaWRlIGltYWdlCiAgICAgICAgICAgIGpzb25fcGF0aCA9IG9zLnBhdGguc3BsaXRleHQoc3RyKGltZ19wYXRoKSlbMF0gKyAnLmpzb24nCiAgICAgICAgICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhqc29uX3BhdGgpOgogICAgICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJMYWJlbCBKU09OIG5vdCBmb3VuZCBmb3IgaW1hZ2U6IHtpbWdfcGF0aH0iKQoKICAgICAgICAjIExvYWQgSlNPTiBhbmQgZXh0cmFjdCB0cnVlIGltYWdlIGNhbnZhcyBkaW1lbnNpb25zCiAgICAgICAgd2l0aCBvcGVuKGpzb25fcGF0aCwgJ3InKSBhcyBmOgogICAgICAgICAgICBkYXRhID0ganNvbi5sb2FkKGYpCgogICAgICAgICMgRHluYW1pYyBjYW52YXMgc2l6ZTogZ3VhcmFudGVlcyBQYXRpZW50IDMyIDRLICgyMTYweDM4NDApIGlzIG5ldmVyIHRydW5jYXRlZAogICAgICAgIGhlaWdodCA9IGRhdGEuZ2V0KCdpbWFnZUhlaWdodCcsIDEwODApCiAgICAgICAgd2lkdGggPSBkYXRhLmdldCgnaW1hZ2VXaWR0aCcsIDE5MjApCiAgICAgICAgY2FudmFzID0gbnAuemVyb3MoKGhlaWdodCwgd2lkdGgpLCBkdHlwZT1ucC51aW50OCkKCiAgICAgICAgIyBEcmF3IGNvbnRvdXJzIHdpdGggdGhpY2tuZXNzIDM1IChleGFjdCBwYXBlciBzdGFuZGFyZCkKICAgICAgICBmb3Igc2hhcGUgaW4gZGF0YS5nZXQoJ3NoYXBlcycsIFtdKToKICAgICAgICAgICAgcG9pbnRzID0gc2hhcGUuZ2V0KCdwb2ludHMnLCBbXSkKICAgICAgICAgICAgbGFiZWwgPSBzaGFwZS5nZXQoJ2xhYmVsJywgJycpLmxvd2VyKCkuc3RyaXAoKQoKICAgICAgICAgICAgIyBDbGFzcyBtYXBwaW5nOiAxID0gUmlkZ2UsIDIgPSBTaWxob3VldHRlLCAzID0gRmFsY2lmb3JtIExpZ2FtZW50CiAgICAgICAgICAgIGlmIGxhYmVsLnN0YXJ0c3dpdGgoJ3InKToKICAgICAgICAgICAgICAgIGNvbG9yID0gMQogICAgICAgICAgICBlbGlmIGxhYmVsLnN0YXJ0c3dpdGgoJ3MnKToKICAgICAgICAgICAgICAgIGNvbG9yID0gMgogICAgICAgICAgICBlbGlmIGxhYmVsLnN0YXJ0c3dpdGgoJ2wnKToKICAgICAgICAgICAgICAgIGNvbG9yID0gMwogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgY29sb3IgPSAwCgogICAgICAgICAgICBmb3IgaSBpbiByYW5nZSgxLCBsZW4ocG9pbnRzKSk6CiAgICAgICAgICAgICAgICBwdDEgPSB0dXBsZShtYXAoaW50LCBwb2ludHNbaSAtIDFdKSkKICAgICAgICAgICAgICAgIHB0MiA9IHR1cGxlKG1hcChpbnQsIHBvaW50c1tpXSkpCiAgICAgICAgICAgICAgICBjdjIubGluZShjYW52YXMsIHB0MSwgcHQyLCBjb2xvciwgMzUpCgogICAgICAgICMgUmVzaXplIHRvIG5ldHdvcmsgaW5wdXQgcmVzb2x1dGlvbiAoMTAyNCwgMTAyNCkgdXNpbmcgSU5URVJfTkVBUkVTVAogICAgICAgIGNhbnZhcyA9IGN2Mi5yZXNpemUoY2FudmFzLCAoMTAyNCwgMTAyNCksIGludGVycG9sYXRpb249Y3YyLklOVEVSX05FQVJFU1QpCgogICAgICAgICMgT25lLWhvdCBtYXAgaW50byA0IGNoYW5uZWxzOiAoMDogQkcsIDE6IFJpZGdlLCAyOiBTaWxob3VldHRlLCAzOiBGYWxjaWZvcm0pCiAgICAgICAgbWFza3MgPSBucC56ZXJvcygoNCwgMTAyNCwgMTAyNCksIGR0eXBlPW5wLnVpbnQ4KQogICAgICAgIG1hc2tzWzBdW2NhbnZhcyA9PSAwXSA9IDI1NQogICAgICAgIG1hc2tzWzFdW2NhbnZhcyA9PSAxXSA9IDI1NQogICAgICAgIG1hc2tzWzJdW2NhbnZhcyA9PSAyXSA9IDI1NQogICAgICAgIG1hc2tzWzNdW2NhbnZhcyA9PSAzXSA9IDI1NQoKICAgICAgICByZXR1cm4gbWFza3MK'))

# 3. Deploy metrics.py
with open('/kaggle/working/experiments/EXPERIMENT_1/utils/metrics.py', 'wb') as f:
    f.write(base64.b64decode('aW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBjdjIKaW1wb3J0IHRvcmNoCgoKZGVmIGNvbXB1dGVfZGljZV9pb3UocHJlZF9iaW5hcnksIGd0X2JpbmFyeSk6CiAgICAiIiIKICAgIENvbXB1dGVzIERpY2UgU2ltaWxhcml0eSBDb2VmZmljaWVudCBhbmQgSW9VIGZvciBiaW5hcnkgMUQgb3IgMkQgYXJyYXlzLgogICAgIiIiCiAgICBpbnRlcnNlY3Rpb24gPSBucC5sb2dpY2FsX2FuZChwcmVkX2JpbmFyeSwgZ3RfYmluYXJ5KS5zdW0oKQogICAgcHJlZF9zdW0gPSBwcmVkX2JpbmFyeS5zdW0oKQogICAgZ3Rfc3VtID0gZ3RfYmluYXJ5LnN1bSgpCiAgICB0b3RhbF9zdW0gPSBwcmVkX3N1bSArIGd0X3N1bQoKICAgIGlmIHRvdGFsX3N1bSA9PSAwOgogICAgICAgIHJldHVybiAxLjAsIDEuMCAgIyBQZXJmZWN0IG1hdGNoIG9uIGVtcHR5IGdyb3VuZCB0cnV0aAogICAgaWYgcHJlZF9zdW0gPT0gMCBvciBndF9zdW0gPT0gMDoKICAgICAgICByZXR1cm4gMC4wLCAwLjAKCiAgICBkaWNlID0gKDIuMCAqIGludGVyc2VjdGlvbikgLyAodG90YWxfc3VtICsgMWUtNykKICAgIGlvdSA9IGludGVyc2VjdGlvbiAvIChwcmVkX3N1bSArIGd0X3N1bSAtIGludGVyc2VjdGlvbiArIDFlLTcpCiAgICByZXR1cm4gZmxvYXQoZGljZSksIGZsb2F0KGlvdSkKCgpkZWYgY29tcHV0ZV9hc3NkKHByZWRfbWFzaywgZ3RfbWFzaywgZmFsbGJhY2s9ODAuMCk6CiAgICAiIiIKICAgIENvbXB1dGVzIEF2ZXJhZ2UgU3ltbWV0cmljIFN1cmZhY2UgRGlzdGFuY2UgKEFTU0QpIGluIHBpeGVscy4KICAgIFVzZXMgc3VyZmFjZV9kaXN0YW5jZSAvIG1lZHB5IGlmIGF2YWlsYWJsZSwgb3Igcm9idXN0IE9wZW5DViBFdWNsaWRlYW4gZGlzdGFuY2UgdHJhbnNmb3JtIGZhbGxiYWNrLgogICAgIiIiCiAgICBwcmVkX21hc2sgPSAocHJlZF9tYXNrID4gMCkuYXN0eXBlKG5wLnVpbnQ4KQogICAgZ3RfbWFzayA9IChndF9tYXNrID4gMCkuYXN0eXBlKG5wLnVpbnQ4KQoKICAgIGlmIHByZWRfbWFzay5zdW0oKSA9PSAwIG9yIGd0X21hc2suc3VtKCkgPT0gMDoKICAgICAgICByZXR1cm4gZmxvYXQoZmFsbGJhY2spCgogICAgIyBUcnkgc3VyZmFjZV9kaXN0YW5jZSAvIG1lZHB5IGZpcnN0CiAgICB0cnk6CiAgICAgICAgZnJvbSBzdXJmYWNlX2Rpc3RhbmNlIGltcG9ydCBtZXRyaWNzCiAgICAgICAgc2QgPSBtZXRyaWNzLmNvbXB1dGVfc3VyZmFjZV9kaXN0YW5jZXMoZ3RfbWFzay5hc3R5cGUoYm9vbCksIHByZWRfbWFzay5hc3R5cGUoYm9vbCksICgxLjAsIDEuMCkpCiAgICAgICAgYXNzZF92YWwgPSBtZXRyaWNzLmNvbXB1dGVfYXZlcmFnZV9zdXJmYWNlX2Rpc3RhbmNlKHNkKVsxXQogICAgICAgIGlmIG5wLmlzbmFuKGFzc2RfdmFsKSBvciBhc3NkX3ZhbCA+IDUwMDoKICAgICAgICAgICAgcmV0dXJuIGZsb2F0KGZhbGxiYWNrKQogICAgICAgIHJldHVybiBmbG9hdChhc3NkX3ZhbCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcGFzcwoKICAgIHRyeToKICAgICAgICBpbXBvcnQgbWVkcHkubWV0cmljCiAgICAgICAgcmV0dXJuIGZsb2F0KG1lZHB5Lm1ldHJpYy5hc3NkKHByZWRfbWFzaywgZ3RfbWFzaykpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKCiAgICAjIFB1cmUgT3BlbkNWIEV1Y2xpZGVhbiBEaXN0YW5jZSBUcmFuc2Zvcm0gRmFsbGJhY2sKICAgICMgRXh0cmFjdCAxLXBpeGVsIGJvdW5kYXJ5IGNvbnRvdXJzCiAgICBjb250b3Vyc19wcmVkLCBfID0gY3YyLmZpbmRDb250b3VycyhwcmVkX21hc2ssIGN2Mi5SRVRSX0xJU1QsIGN2Mi5DSEFJTl9BUFBST1hfTk9ORSkKICAgIGNvbnRvdXJzX2d0LCBfID0gY3YyLmZpbmRDb250b3VycyhndF9tYXNrLCBjdjIuUkVUUl9MSVNULCBjdjIuQ0hBSU5fQVBQUk9YX05PTkUpCgogICAgYm9yZGVyX3ByZWQgPSBucC56ZXJvc19saWtlKHByZWRfbWFzaykKICAgIGJvcmRlcl9ndCA9IG5wLnplcm9zX2xpa2UoZ3RfbWFzaykKCiAgICBjdjIuZHJhd0NvbnRvdXJzKGJvcmRlcl9wcmVkLCBjb250b3Vyc19wcmVkLCAtMSwgMSwgMSkKICAgIGN2Mi5kcmF3Q29udG91cnMoYm9yZGVyX2d0LCBjb250b3Vyc19ndCwgLTEsIDEsIDEpCgogICAgIyBEaXN0YW5jZSB0byBHVCBzdXJmYWNlCiAgICBkaXN0X3RvX2d0ID0gY3YyLmRpc3RhbmNlVHJhbnNmb3JtKDEgLSBib3JkZXJfZ3QsIGN2Mi5ESVNUX0wyLCA1KQogICAgIyBEaXN0YW5jZSB0byBQcmVkIHN1cmZhY2UKICAgIGRpc3RfdG9fcHJlZCA9IGN2Mi5kaXN0YW5jZVRyYW5zZm9ybSgxIC0gYm9yZGVyX3ByZWQsIGN2Mi5ESVNUX0wyLCA1KQoKICAgIGRpc3RfcHJlZF90b19ndCA9IGRpc3RfdG9fZ3RbYm9yZGVyX3ByZWQgPT0gMV0KICAgIGRpc3RfZ3RfdG9fcHJlZCA9IGRpc3RfdG9fcHJlZFtib3JkZXJfZ3QgPT0gMV0KCiAgICBpZiBsZW4oZGlzdF9wcmVkX3RvX2d0KSA9PSAwIG9yIGxlbihkaXN0X2d0X3RvX3ByZWQpID09IDA6CiAgICAgICAgcmV0dXJuIGZsb2F0KGZhbGxiYWNrKQoKICAgIGFzc2RfdmFsID0gKGRpc3RfcHJlZF90b19ndC5tZWFuKCkgKyBkaXN0X2d0X3RvX3ByZWQubWVhbigpKSAvIDIuMAogICAgcmV0dXJuIGZsb2F0KG1pbihhc3NkX3ZhbCwgZmFsbGJhY2spKQoKCmRlZiBldmFsdWF0ZV9iYXRjaChwcmVkX2xvZ2l0cywgZ3RfbWFza3MpOgogICAgIiIiCiAgICBFdmFsdWF0ZXMgYSBiYXRjaCBvZiBtdWx0aS1jbGFzcyBwcmVkaWN0aW9ucyBhZ2FpbnN0IGdyb3VuZCB0cnV0aC4KICAgIHByZWRfbG9naXRzOiBUZW5zb3Igb2Ygc2hhcGUgKEIsIDQsIEgsIFcpCiAgICBndF9tYXNrczogVGVuc29yIG9mIHNoYXBlIChCLCA0LCBILCBXKSB3aGVyZSBtYXNrcyBhcmUgb25lLWhvdCAoMDogQkcsIDE6IFJpZGdlLCAyOiBTaWwsIDM6IEZhbGMpCiAgICAKICAgIFJldHVybnMgbGlzdCBvZiBtZXRyaWMgZGljdHMgcGVyIHNhbXBsZS4KICAgICIiIgogICAgcHJlZF9jbGFzc2VzID0gdG9yY2guYXJnbWF4KHByZWRfbG9naXRzLCBkaW09MSkuZGV0YWNoKCkuY3B1KCkubnVtcHkoKSAgIyAoQiwgSCwgVykKICAgIGd0X2NsYXNzZXMgPSB0b3JjaC5hcmdtYXgoZ3RfbWFza3MsIGRpbT0xKS5kZXRhY2goKS5jcHUoKS5udW1weSgpICAgICAgICAjIChCLCBILCBXKQoKICAgIGJhdGNoX21ldHJpY3MgPSBbXQoKICAgIGZvciBiIGluIHJhbmdlKHByZWRfY2xhc3Nlcy5zaGFwZVswXSk6CiAgICAgICAgcF9tYXAgPSBwcmVkX2NsYXNzZXNbYl0KICAgICAgICBnX21hcCA9IGd0X2NsYXNzZXNbYl0KCiAgICAgICAgIyBQZXItY2xhc3MgZm9yZWdyb3VuZCBtZXRyaWNzICgxOiBSaWRnZSwgMjogU2lsaG91ZXR0ZSwgMzogRmFsY2lmb3JtKQogICAgICAgIGNsYXNzX2RpY2VzID0gW10KICAgICAgICBjbGFzc19pb3VzID0gW10KICAgICAgICBjbGFzc19hc3NkcyA9IFtdCgogICAgICAgIGZvciBjLCBuYW1lIGluIGVudW1lcmF0ZShbJ3JpZGdlJywgJ3NpbGhvdWV0dGUnLCAnZmFsY2lmb3JtJ10sIHN0YXJ0PTEpOgogICAgICAgICAgICBwX2MgPSAocF9tYXAgPT0gYykKICAgICAgICAgICAgZ19jID0gKGdfbWFwID09IGMpCgogICAgICAgICAgICBkLCBpb3UgPSBjb21wdXRlX2RpY2VfaW91KHBfYywgZ19jKQogICAgICAgICAgICBjbGFzc19kaWNlcy5hcHBlbmQoZCkKICAgICAgICAgICAgY2xhc3NfaW91cy5hcHBlbmQoaW91KQoKICAgICAgICAgICAgaWYgZ19jLnN1bSgpID4gMDoKICAgICAgICAgICAgICAgIGFzc2RfYyA9IGNvbXB1dGVfYXNzZChwX2MsIGdfYykKICAgICAgICAgICAgICAgIGNsYXNzX2Fzc2RzLmFwcGVuZChhc3NkX2MpCgogICAgICAgIG1hY3JvX2RpY2UgPSBmbG9hdChucC5tZWFuKGNsYXNzX2RpY2VzKSkKICAgICAgICBtYWNyb19pb3UgPSBmbG9hdChucC5tZWFuKGNsYXNzX2lvdXMpKQogICAgICAgIG1hY3JvX2Fzc2QgPSBmbG9hdChucC5tZWFuKGNsYXNzX2Fzc2RzKSkgaWYgbGVuKGNsYXNzX2Fzc2RzKSA+IDAgZWxzZSA4MC4wCgogICAgICAgICMgT3ZlcmFsbCBmbGF0dGVuZWQgZm9yZWdyb3VuZCBtZXRyaWMgKGV4YWN0IHJlcG9zL1RvcG9OZXQvdGVzdC5weSBzdGFuZGFyZCkKICAgICAgICBwX2ZnID0gKHBfbWFwID4gMCkKICAgICAgICBnX2ZnID0gKGdfbWFwID4gMCkKICAgICAgICBmZ19kaWNlLCBmZ19pb3UgPSBjb21wdXRlX2RpY2VfaW91KHBfZmcsIGdfZmcpCiAgICAgICAgZmdfYXNzZCA9IGNvbXB1dGVfYXNzZChwX2ZnLCBnX2ZnKQoKICAgICAgICBiYXRjaF9tZXRyaWNzLmFwcGVuZCh7CiAgICAgICAgICAgICdtYWNyb19kaWNlJzogbWFjcm9fZGljZSwKICAgICAgICAgICAgJ21hY3JvX2lvdSc6IG1hY3JvX2lvdSwKICAgICAgICAgICAgJ21hY3JvX2Fzc2QnOiBtYWNyb19hc3NkLAogICAgICAgICAgICAnZmdfZGljZSc6IGZnX2RpY2UsCiAgICAgICAgICAgICdmZ19pb3UnOiBmZ19pb3UsCiAgICAgICAgICAgICdmZ19hc3NkJzogZmdfYXNzZCwKICAgICAgICAgICAgJ3JpZGdlX2RpY2UnOiBjbGFzc19kaWNlc1swXSwKICAgICAgICAgICAgJ3NpbF9kaWNlJzogY2xhc3NfZGljZXNbMV0sCiAgICAgICAgICAgICdmYWxjX2RpY2UnOiBjbGFzc19kaWNlc1syXSwKICAgICAgICAgICAgJ3JpZGdlX2lvdSc6IGNsYXNzX2lvdXNbMF0sCiAgICAgICAgICAgICdzaWxfaW91JzogY2xhc3NfaW91c1sxXSwKICAgICAgICAgICAgJ2ZhbGNfaW91JzogY2xhc3NfaW91c1syXSwKICAgICAgICB9KQoKICAgIHJldHVybiBiYXRjaF9tZXRyaWNzCg=='))

# 4. Deploy toponet_ablation.py
with open('/kaggle/working/experiments/EXPERIMENT_1/models/toponet_ablation.py', 'wb') as f:
    f.write(base64.b64decode('aW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgojIEVuc3VyZSByZXBvcy9Ub3BvTmV0IGlzIGluIHN5cy5wYXRoClJFUE9fUk9PVCA9IG9zLnBhdGguYWJzcGF0aChvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKF9fZmlsZV9fKSwgJy4uLy4uLy4uL3JlcG9zL1RvcG9OZXQnKSkKaWYgUkVQT19ST09UIG5vdCBpbiBzeXMucGF0aDoKICAgIHN5cy5wYXRoLmluc2VydCgwLCBSRVBPX1JPT1QpCgpmcm9tIGRlcHRoX2FueXRoaW5nX3YyLmRwdCBpbXBvcnQgRGVwdGhBbnl0aGluZ1YyCmZyb20gbW9kZWxzLnJlc25ldCBpbXBvcnQgUmVzTmV0MzQKZnJvbSBtb2RlbHMuY29udGV4dF9tb2R1bGVzIGltcG9ydCBnZXRfY29udGV4dF9tb2R1bGUKZnJvbSBtb2RlbHMubW9kZWxfdXRpbHMgaW1wb3J0IENvbnZCTkFjdCwgU3dpc2gKZnJvbSBtb2RlbHMuZGVjb2RlciBpbXBvcnQgRGVjb2Rlcgpmcm9tIG1vZGVscy5iZWZ1c2lvbiBpbXBvcnQgQmVGdXNpb24KZnJvbSBEU0NOZXQuZHNfZW5jb2RlciBpbXBvcnQgRFNDTmV0X0VuY29kZXIKCgpkZWYgX3NhZmVfaW50ZXJwb2xhdGVfYXJlYSh4LCBzaXplKToKICAgICIiIkFyZWEgaW50ZXJwb2xhdGlvbiB3aXRoIGF1dG9tYXRpYyBNUFMgQ1BVIGZhbGxiYWNrIGZvciBub24tZGl2aXNpYmxlIHNpemVzLiIiIgogICAgaWYgeC5kZXZpY2UudHlwZSA9PSAnbXBzJzoKICAgICAgICByZXR1cm4gRi5pbnRlcnBvbGF0ZSh4LmNwdSgpLCBzaXplPXNpemUsIG1vZGU9J2FyZWEnKS50byh4LmRldmljZSkKICAgIHJldHVybiBGLmludGVycG9sYXRlKHgsIHNpemU9c2l6ZSwgbW9kZT0nYXJlYScpCgoKY2xhc3MgU2ltcGxlQ29uY2F0RnVzaW9uKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFNpbXBsZSBjb25jYXRlbmF0aW9uIGJhc2VsaW5lIHJlcGxhY2luZyBCVEYgKEJvdW5kYXJ5LUF3YXJlIFRvcG9sb2dpY2FsIEZ1c2lvbikuCiAgICBNZXJnZXMgUkdCIGFuZCBkZXB0aCBmZWF0dXJlIG1hcHMgdmlhIDF4MSBjb252b2x1dGlvbi4KICAgICIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoYW5uZWxzKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbnYgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoaW5fY2hhbm5lbHMgKiAyLCBpbl9jaGFubmVscywga2VybmVsX3NpemU9MSwgYmlhcz1GYWxzZSksCiAgICAgICAgICAgIG5uLkJhdGNoTm9ybTJkKGluX2NoYW5uZWxzKSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpCiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHJnYl9mZWF0LCBkZXB0aF9mZWF0LCBwcmV2X2ZlYXQ9Tm9uZSk6CiAgICAgICAgaWYgcHJldl9mZWF0IGlzIG5vdCBOb25lIGFuZCBwcmV2X2ZlYXQuc2hhcGVbMjpdICE9IHJnYl9mZWF0LnNoYXBlWzI6XToKICAgICAgICAgICAgcHJldl9mZWF0ID0gRi5pbnRlcnBvbGF0ZShwcmV2X2ZlYXQsIHNpemU9cmdiX2ZlYXQuc2hhcGVbMjpdLCBtb2RlPSdiaWxpbmVhcicsIGFsaWduX2Nvcm5lcnM9RmFsc2UpCiAgICAgICAgZnVzZWQgPSBzZWxmLmNvbnYodG9yY2guY2F0KFtyZ2JfZmVhdCwgZGVwdGhfZmVhdF0sIGRpbT0xKSkKICAgICAgICBpZiBwcmV2X2ZlYXQgaXMgbm90IE5vbmUgYW5kIHByZXZfZmVhdC5zaGFwZVsxXSA9PSBmdXNlZC5zaGFwZVsxXToKICAgICAgICAgICAgZnVzZWQgPSBmdXNlZCArIHByZXZfZmVhdAogICAgICAgIHJldHVybiBmdXNlZCwgZnVzZWQKCgpjbGFzcyBUb3BvTmV0QWJsYXRpb25Nb2RlbChubi5Nb2R1bGUpOgogICAgIiIiCiAgICBVbmlmaWVkIFRvcG9OZXQgTW9kZWwgc3VwcG9ydGluZyBhbGwgNiBvZmZpY2lhbCBwYXBlciBhYmxhdGlvbiBtb2RlczoKICAgICAgMS4gJ2Z1bGwnOiBGdWxsIFRvcG9OZXQgKFNuYWtlIERTQ05ldCArIEJURiArIGNsRGljZSArIEJldHRpKQogICAgICAyLiAnYmFzZWxpbmUnOiBTdGFuZGFyZCBDb252ICsgU2ltcGxlIENvbmNhdCAoTm8gQlRGLCBubyB0b3BvIGxvc3NlcykKICAgICAgMy4gJ3dvX2xwZXInOiBTbmFrZSBEU0NOZXQgKyBCVEYgKyBTb2Z0IERpY2UgKyBjbERpY2UgKG5vIEJldHRpKQogICAgICA0LiAnd29fbGNsJzogU25ha2UgRFNDTmV0ICsgQlRGICsgU29mdCBEaWNlICsgQmV0dGkgKG5vIGNsRGljZSkKICAgICAgNS4gJ3dvX2xwZXJfbGNsJzogU25ha2UgRFNDTmV0ICsgQlRGICsgU29mdCBEaWNlIG9ubHkgKG5vIHRvcG8gbG9zcykKICAgICAgNi4gJ3dvX2J0Zic6IFNuYWtlIERTQ05ldCArIFNpbXBsZSBDb25jYXQgKyBTb2Z0IERpY2UgKyBjbERpY2UgKyBCZXR0aQogICAgIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgYWJsYXRpb25fbW9kZT0nZnVsbCcsIGRlcHRoX3BhdGg9Tm9uZSwgbnVtX2NsYXNzZXM9NCwgaGVpZ2h0PTEwMjQsIHdpZHRoPTEwMjQpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYWJsYXRpb25fbW9kZSA9IGFibGF0aW9uX21vZGUKICAgICAgICBzZWxmLmRlcHRoX3BhdGggPSBkZXB0aF9wYXRoCgogICAgICAgICMgMS4gRGVwdGggQW55dGhpbmcgVjIgRm91bmRhdGlvbiBNb2RlbAogICAgICAgIGRlcHRoX2NvbmZpZ3MgPSB7CiAgICAgICAgICAgICd2aXRiJzogeydlbmNvZGVyJzogJ3ZpdGInLCAnZmVhdHVyZXMnOiAxMjgsICdvdXRfY2hhbm5lbHMnOiBbOTYsIDE5MiwgMzg0LCA3NjhdfQogICAgICAgIH0KICAgICAgICBzZWxmLmRlcHRoX2VuY29kZXIgPSBEZXB0aEFueXRoaW5nVjIoKipkZXB0aF9jb25maWdzWyd2aXRiJ10pCiAgICAgICAgaWYgZGVwdGhfcGF0aCBhbmQgb3MucGF0aC5leGlzdHMoZGVwdGhfcGF0aCk6CiAgICAgICAgICAgIHN0YXRlX2RpY3QgPSB0b3JjaC5sb2FkKGRlcHRoX3BhdGgsIG1hcF9sb2NhdGlvbj0nY3B1JykKICAgICAgICAgICAgc2VsZi5kZXB0aF9lbmNvZGVyLmxvYWRfc3RhdGVfZGljdChzdGF0ZV9kaWN0KQogICAgICAgICAgICBwcmludChmIuKchSBMb2FkZWQgRGVwdGggQW55dGhpbmcgVjIgd2VpZ2h0cyBmcm9tOiB7ZGVwdGhfcGF0aH0iKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHByaW50KGYi4pqg77iPICBEZXB0aCB3ZWlnaHRzIG5vdCBmb3VuZCBhdCAne2RlcHRoX3BhdGh9Jy4gSW5pdGlhbGl6ZWQgd2l0aCByYW5kb20gd2VpZ2h0cyAoT0sgZm9yIHNtb2tlIHRlc3RzKS4iKQogICAgICAgIHNlbGYuZGVwdGhfZW5jb2Rlci5ldmFsKCkKICAgICAgICBzZWxmLmRlcHRoX2VuY29kZXIucmVxdWlyZXNfZ3JhZF8oRmFsc2UpCgogICAgICAgICMgMi4gRGVwdGggRmVhdHVyZSBFeHRyYWN0b3IgKFNuYWtlIERTQ05ldCB2cyBTdGFuZGFyZCBDb252KQogICAgICAgIHNlbGYudXNlX3NuYWtlID0gKGFibGF0aW9uX21vZGUgIT0gJ2Jhc2VsaW5lJykKICAgICAgICBpZiBzZWxmLnVzZV9zbmFrZToKICAgICAgICAgICAgc2VsZi5kc2NfZW5jb2RlciA9IERTQ05ldF9FbmNvZGVyKCkKICAgICAgICBlbHNlOgogICAgICAgICAgICAjIEJhc2VsaW5lIHVzZXMgc3RhbmRhcmQgUmVzTmV0IGJsb2NrcyBmb3IgZGVwdGgKICAgICAgICAgICAgc2VsZi5kc2NfZW5jb2RlciA9IFJlc05ldDM0KGlucHV0X2NoYW5uZWxzPTMsIHByZXRyYWluZWRfb25faW1hZ2VuZXQ9RmFsc2UpCgogICAgICAgICMgMy4gUkdCIEVuY29kZXIgKFJlc05ldC0zNCkKICAgICAgICBzZWxmLnJnYl9lbmNvZGVyID0gUmVzTmV0MzQoaW5wdXRfY2hhbm5lbHM9MywgcHJldHJhaW5lZF9vbl9pbWFnZW5ldD1GYWxzZSkKCiAgICAgICAgIyA0LiBNdWx0aS1Nb2RhbCBGdXNpb24gKEJURiB2cyBTaW1wbGUgQ29uY2F0KQogICAgICAgIHNlbGYudXNlX2J0ZiA9IChhYmxhdGlvbl9tb2RlIG5vdCBpbiBbJ2Jhc2VsaW5lJywgJ3dvX2J0ZiddKQogICAgICAgIGlmIHNlbGYudXNlX2J0ZjoKICAgICAgICAgICAgc2VsZi5iZTAgPSBCZUZ1c2lvbig2NCwgNTEyLCA1MTIsIGlzRmlyc3Q9VHJ1ZSkKICAgICAgICAgICAgc2VsZi5iZTEgPSBCZUZ1c2lvbihzZWxmLnJnYl9lbmNvZGVyLmRvd25fNF9jaGFubmVsc19vdXQsIDI1NiwgMjU2KQogICAgICAgICAgICBzZWxmLmJlMiA9IEJlRnVzaW9uKHNlbGYucmdiX2VuY29kZXIuZG93bl84X2NoYW5uZWxzX291dCwgMTI4LCAxMjgpCiAgICAgICAgICAgIHNlbGYuYmUzID0gQmVGdXNpb24oc2VsZi5yZ2JfZW5jb2Rlci5kb3duXzE2X2NoYW5uZWxzX291dCwgNjQsIDY0KQogICAgICAgICAgICBzZWxmLmJlNCA9IEJlRnVzaW9uKHNlbGYucmdiX2VuY29kZXIuZG93bl8zMl9jaGFubmVsc19vdXQsIDMyLCAzMiwgaXNMYXN0PVRydWUpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5iZTAgPSBTaW1wbGVDb25jYXRGdXNpb24oNjQpCiAgICAgICAgICAgIHNlbGYuYmUxID0gU2ltcGxlQ29uY2F0RnVzaW9uKHNlbGYucmdiX2VuY29kZXIuZG93bl80X2NoYW5uZWxzX291dCkKICAgICAgICAgICAgc2VsZi5iZTIgPSBTaW1wbGVDb25jYXRGdXNpb24oc2VsZi5yZ2JfZW5jb2Rlci5kb3duXzhfY2hhbm5lbHNfb3V0KQogICAgICAgICAgICBzZWxmLmJlMyA9IFNpbXBsZUNvbmNhdEZ1c2lvbihzZWxmLnJnYl9lbmNvZGVyLmRvd25fMTZfY2hhbm5lbHNfb3V0KQogICAgICAgICAgICBzZWxmLmJlNCA9IFNpbXBsZUNvbmNhdEZ1c2lvbihzZWxmLnJnYl9lbmNvZGVyLmRvd25fMzJfY2hhbm5lbHNfb3V0KQoKICAgICAgICAjIFNraXAgY29ubmVjdGlvbnMKICAgICAgICBjaGFubmVsc19kZWNvZGVyID0gWzEyOCwgMTI4LCAxMjhdCiAgICAgICAgc2VsZi5za2lwX2xheWVyMSA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIENvbnZCTkFjdChzZWxmLnJnYl9lbmNvZGVyLmRvd25fNF9jaGFubmVsc19vdXQsIGNoYW5uZWxzX2RlY29kZXJbMl0sIGtlcm5lbF9zaXplPTEsIGFjdGl2YXRpb249bm4uUmVMVShpbnBsYWNlPVRydWUpKQogICAgICAgICkKICAgICAgICBzZWxmLnNraXBfbGF5ZXIyID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgQ29udkJOQWN0KHNlbGYucmdiX2VuY29kZXIuZG93bl84X2NoYW5uZWxzX291dCwgY2hhbm5lbHNfZGVjb2RlclsxXSwga2VybmVsX3NpemU9MSwgYWN0aXZhdGlvbj1ubi5SZUxVKGlucGxhY2U9VHJ1ZSkpCiAgICAgICAgKQogICAgICAgIHNlbGYuc2tpcF9sYXllcjMgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBDb252Qk5BY3Qoc2VsZi5yZ2JfZW5jb2Rlci5kb3duXzE2X2NoYW5uZWxzX291dCwgY2hhbm5lbHNfZGVjb2RlclswXSwga2VybmVsX3NpemU9MSwgYWN0aXZhdGlvbj1ubi5SZUxVKGlucGxhY2U9VHJ1ZSkpCiAgICAgICAgKQoKICAgICAgICAjIENvbnRleHQgTW9kdWxlICYgRGVjb2RlcgogICAgICAgIHNlbGYuY29udGV4dF9tb2R1bGUsIGNoYW5uZWxzX2FmdGVyX2NvbnRleHQgPSBnZXRfY29udGV4dF9tb2R1bGUoCiAgICAgICAgICAgICdwcG0nLCBzZWxmLnJnYl9lbmNvZGVyLmRvd25fMzJfY2hhbm5lbHNfb3V0LCBjaGFubmVsc19kZWNvZGVyWzBdLAogICAgICAgICAgICBpbnB1dF9zaXplPShoZWlnaHQgLy8gMzIsIHdpZHRoIC8vIDMyKSwgYWN0aXZhdGlvbj1ubi5SZUxVKGlucGxhY2U9VHJ1ZSksIHVwc2FtcGxpbmdfbW9kZT0nYmlsaW5lYXInCiAgICAgICAgKQoKICAgICAgICBzZWxmLmRlY29kZXIgPSBEZWNvZGVyKAogICAgICAgICAgICBjaGFubmVsc19pbj1jaGFubmVsc19hZnRlcl9jb250ZXh0LCBjaGFubmVsc19kZWNvZGVyPWNoYW5uZWxzX2RlY29kZXIsCiAgICAgICAgICAgIGFjdGl2YXRpb249bm4uUmVMVShpbnBsYWNlPVRydWUpLCBucl9kZWNvZGVyX2Jsb2Nrcz1bMSwgMSwgMV0sCiAgICAgICAgICAgIGVuY29kZXJfZGVjb2Rlcl9mdXNpb249J2FkZCcsIHVwc2FtcGxpbmdfbW9kZT0nYmlsaW5lYXInLCBudW1fY2xhc3Nlcz1udW1fY2xhc3NlcwogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBpbWFnZSk6CiAgICAgICAgIyAxLiBPbi10aGUtZmx5IGRlcHRoIGVzdGltYXRpb24gYXQgKDEwMjIsIDEwMjIpIFttdWx0aXBsZSBvZiBWaVQgcGF0Y2ggc2l6ZSAxNF0KICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgaW1nX2RlcHRoX2luID0gX3NhZmVfaW50ZXJwb2xhdGVfYXJlYShpbWFnZSwgc2l6ZT0oMTAyMiwgMTAyMikpCiAgICAgICAgICAgIHJhd19kZXB0aCA9IHNlbGYuZGVwdGhfZW5jb2Rlci5pbmZlcl9pbWFnZShpbWdfZGVwdGhfaW4pCiAgICAgICAgICAgIGRlcHRoXzNjaCA9IHJhd19kZXB0aC5leHBhbmQoLTEsIDMsIC0xLCAtMSkKCiAgICAgICAgIyAyLiBEZXB0aCBGZWF0dXJlIEV4dHJhY3Rpb24KICAgICAgICBpZiBzZWxmLnVzZV9zbmFrZToKICAgICAgICAgICAgZDAsIGQxLCBkMiwgZDMsIGRlcHRoX291dCA9IHNlbGYuZHNjX2VuY29kZXIoZGVwdGhfM2NoKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgICMgQmFzZWxpbmUgc3RhbmRhcmQgQ05OIGRlcHRoIGVuY29kZXIKICAgICAgICAgICAgb3V0X2QgPSBzZWxmLmRzY19lbmNvZGVyLmZvcndhcmRfZmlyc3RfY29udihkZXB0aF8zY2gpCiAgICAgICAgICAgIGQwID0gb3V0X2QKICAgICAgICAgICAgb3V0X2QgPSBGLm1heF9wb29sMmQob3V0X2QsIGtlcm5lbF9zaXplPTMsIHN0cmlkZT0yLCBwYWRkaW5nPTEpCiAgICAgICAgICAgIGQxID0gc2VsZi5kc2NfZW5jb2Rlci5mb3J3YXJkX2xheWVyMShvdXRfZCkKICAgICAgICAgICAgZDIgPSBzZWxmLmRzY19lbmNvZGVyLmZvcndhcmRfbGF5ZXIyKGQxKQogICAgICAgICAgICBkMyA9IHNlbGYuZHNjX2VuY29kZXIuZm9yd2FyZF9sYXllcjMoZDIpCiAgICAgICAgICAgIGRlcHRoX291dCA9IHNlbGYuZHNjX2VuY29kZXIuZm9yd2FyZF9sYXllcjQoZDMpCgogICAgICAgICMgMy4gUkdCIEZlYXR1cmUgRXh0cmFjdGlvbiAmIFByb2dyZXNzaXZlIE11bHRpLU1vZGFsIEZ1c2lvbgogICAgICAgIG91dF9yZ2IgPSBzZWxmLnJnYl9lbmNvZGVyLmZvcndhcmRfZmlyc3RfY29udihpbWFnZSkKICAgICAgICBza2lwZjAsIG91dF9mMCA9IHNlbGYuYmUwKG91dF9yZ2IsIF9zYWZlX2ludGVycG9sYXRlX2FyZWEoZDAsIHNpemU9b3V0X3JnYi5zaGFwZVsyOl0pKQogICAgICAgIG91dF9yZ2IgPSBGLm1heF9wb29sMmQob3V0X3JnYiwga2VybmVsX3NpemU9Mywgc3RyaWRlPTIsIHBhZGRpbmc9MSkKCiAgICAgICAgIyBCbG9jayAxCiAgICAgICAgb3V0X3JnYiA9IHNlbGYucmdiX2VuY29kZXIuZm9yd2FyZF9sYXllcjEob3V0X3JnYikKICAgICAgICBza2lwZjEsIG91dF9mMSA9IHNlbGYuYmUxKG91dF9yZ2IsIF9zYWZlX2ludGVycG9sYXRlX2FyZWEoZDEsIHNpemU9b3V0X3JnYi5zaGFwZVsyOl0pLCBvdXRfZjApCiAgICAgICAgc2tpcDEgPSBzZWxmLnNraXBfbGF5ZXIxKHNraXBmMSkKCiAgICAgICAgIyBCbG9jayAyCiAgICAgICAgb3V0X3JnYiA9IHNlbGYucmdiX2VuY29kZXIuZm9yd2FyZF9sYXllcjIob3V0X3JnYikKICAgICAgICBza2lwZjIsIG91dF9mMiA9IHNlbGYuYmUyKG91dF9yZ2IsIF9zYWZlX2ludGVycG9sYXRlX2FyZWEoZDIsIHNpemU9b3V0X3JnYi5zaGFwZVsyOl0pLCBvdXRfZjEpCiAgICAgICAgc2tpcDIgPSBzZWxmLnNraXBfbGF5ZXIyKHNraXBmMikKCiAgICAgICAgIyBCbG9jayAzCiAgICAgICAgb3V0X3JnYiA9IHNlbGYucmdiX2VuY29kZXIuZm9yd2FyZF9sYXllcjMob3V0X3JnYikKICAgICAgICBza2lwZjMsIG91dF9mMyA9IHNlbGYuYmUzKG91dF9yZ2IsIF9zYWZlX2ludGVycG9sYXRlX2FyZWEoZDMsIHNpemU9b3V0X3JnYi5zaGFwZVsyOl0pLCBvdXRfZjIpCiAgICAgICAgc2tpcDMgPSBzZWxmLnNraXBfbGF5ZXIzKHNraXBmMykKCiAgICAgICAgIyBCbG9jayA0CiAgICAgICAgb3V0X3JnYiA9IHNlbGYucmdiX2VuY29kZXIuZm9yd2FyZF9sYXllcjQob3V0X3JnYikKICAgICAgICBza2lwZjQsIG91dF9mNCA9IHNlbGYuYmU0KG91dF9yZ2IsIF9zYWZlX2ludGVycG9sYXRlX2FyZWEoZGVwdGhfb3V0LCBzaXplPW91dF9yZ2Iuc2hhcGVbMjpdKSwgb3V0X2YzKQoKICAgICAgICAjIENvbnRleHQgTW9kdWxlICYgRGVjb2RlcgogICAgICAgIGNvbnRleHRfb3V0ID0gc2VsZi5jb250ZXh0X21vZHVsZShvdXRfZjQpCiAgICAgICAgZGVjb2Rlcl9vdXRzLCBfID0gc2VsZi5kZWNvZGVyKGVuY19vdXRzPVtjb250ZXh0X291dCwgc2tpcDMsIHNraXAyLCBza2lwMV0pCiAgICAgICAgbG9naXRzID0gRi5sb2dfc29mdG1heChkZWNvZGVyX291dHMsIGRpbT0xKQoKICAgICAgICByZXR1cm4gbG9naXRzLCByYXdfZGVwdGgK'))

# 5. Deploy train_toponet.py
with open('/kaggle/working/experiments/EXPERIMENT_1/scripts/train_toponet.py', 'wb') as f:
    f.write(base64.b64decode('...'))

print("✅ All EXPERIMENT_1 modules deployed cleanly without string escaping errors.")


## Step 7: Run TopoNet Ablation Study
Configure and execute the desired ablation runs.

### Configuration Guide:
- `RUN_ALL_ABLATIONS = False`: Runs **Run 1.0 (Full TopoNet)** on both Validation and Test splits first.
- `RUN_ALL_ABLATIONS = True`: Runs all 6 paper ablation configurations sequentially:
  1. `full`: Full TopoNet (ResNet34 + Depth + DSCNet + BTF + clDice + Betti)
  2. `baseline`: Baseline (ResNet34 + Depth + Conv + Simple Concat + Soft Dice)
  3. `wo_lper`: w/o $L_{per}$ (clDice only)
  4. `wo_lcl`: w/o $L_{cl}$ (Betti only)
  5. `wo_lper_lcl`: w/o $L_{per}$ & $L_{cl}$ (Soft Dice only)
  6. `wo_btf`: w/o BTF (Simple Concat + clDice + Betti)


In [ ]:
# ==============================================================================
# 🎛️ EXPERIMENT RUNNER CONFIGURATION
# ==============================================================================
RUN_ALL_ABLATIONS = False    # Set to True to run all 6 paper ablations in sequence
RUN_FULL_TEST_SPLIT = True   # Evaluate Full TopoNet on Test split (Table 1 SOTA)
EPOCHS = 100                 # Standard paper epochs (100)
BATCH_SIZE = 2               # Micro-batch 2 + Accumulation 2 = Effective Batch 4 (fits 16GB T4)
ACCUMULATION_STEPS = 2
LR = 8e-5
WEIGHT_DECAY = 3e-5

if RUN_ALL_ABLATIONS:
    ABLATION_LIST = ['full', 'baseline', 'wo_lper', 'wo_lcl', 'wo_lper_lcl', 'wo_btf']
else:
    # Test Run 1.0 (Full TopoNet) first
    ABLATION_LIST = ['full']

print(f"📋 Ablation queue: {ABLATION_LIST}")
print(f"📊 Training parameters: {EPOCHS} epochs, lr={LR}, effective batch size={BATCH_SIZE * ACCUMULATION_STEPS}")

# Execute runs sequentially
for abl in ABLATION_LIST:
    run_save_dir = f"/kaggle/working/results/run_{abl}"
    os.makedirs(run_save_dir, exist_ok=True)
    
    print("\n" + "=" * 80)
    print(f"🚀 LAUNCHING RUN: {abl.upper()}")
    print(f"📁 Output Directory: {run_save_dir}")
    print("=" * 80)
    
    cmd = [
        "python", "/kaggle/working/experiments/EXPERIMENT_1/scripts/train_toponet.py",
        "--train_dir", train_dir,
        "--val_dir", val_dir,
        "--test_dir", test_dir,
        "--depth_weights", depth_ckpt,
        "--save_dir", run_save_dir,
        "--ablation", abl,
        "--epochs", str(EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--accumulation_steps", str(ACCUMULATION_STEPS),
        "--lr", str(LR),
        "--weight_decay", str(WEIGHT_DECAY),
        "--eval_splits", "both" if (abl == 'full' and RUN_FULL_TEST_SPLIT) else "val"
    ]
    
    subprocess.run(cmd, check=True)


## Step 8: Summary Benchmark Table & Results Download
Collect all metric summaries, verify Patient 40 diagnostic plots, and package into `EXPERIMENT_1_RESULTS.zip`.


In [ ]:
import json
import glob
import pandas as pd
from IPython.display import display, Markdown, FileLink

print("=" * 80)
print("📊 EMPIRICAL ABLATION RESULTS (LIVE FROM EXPERIMENT RUNS)")
print("=" * 80)

summary_files = sorted(glob.glob('/kaggle/working/results/run_*/summary_metrics.json'))

if not summary_files:
    print("⚠️ No summary_metrics.json files found. Have you executed the training cell above?")
else:
    rows = []
    for sf in summary_files:
        try:
            with open(sf, 'r') as f:
                data = json.load(f)
            abl = data.get('ablation_mode', 'unknown').upper()
            val_m = data.get('val_metrics', {})
            test_m = data.get('test_metrics', {})
            
            row = {
                'Ablation Mode': abl,
                'Val Macro DSC': f"{val_m.get('macro_dice', 0)*100:.2f}%",
                'Val FG DSC': f"{val_m.get('fg_dice', 0)*100:.2f}%",
                'Val IoU': f"{val_m.get('macro_iou', 0)*100:.2f}%",
                'Val ASSD (px)': f"{val_m.get('macro_assd', 0):.2f}",
                'Ridge DSC': f"{val_m.get('ridge_dice', 0)*100:.2f}%",
                'Silhouette DSC': f"{val_m.get('sil_dice', 0)*100:.2f}%",
                'Falciform DSC': f"{val_m.get('falc_dice', 0)*100:.2f}%",
                'Patient 40 DSC': f"{val_m.get('patient_40_dice', 0)*100:.2f}%",
                'Test Macro DSC': f"{test_m.get('macro_dice', 0)*100:.2f}%" if test_m else "--",
                'Latency (ms)': f"{val_m.get('mean_latency_ms', 0):.1f}"
            }
            rows.append(row)
        except Exception as e:
            print(f"Error reading {sf}: {e}")

    df_results = pd.DataFrame(rows)
    display(Markdown(df_results.to_markdown(index=False)))

# Check Patient 40 visual diagnostics
p40_images = glob.glob('/kaggle/working/results/**/visualizations_patient40/*.png', recursive=True)
print(f"\n🔍 Patient 40 diagnostic panels generated: {len(p40_images)}")

# Package all results into ZIP
zip_dest = '/kaggle/working/EXPERIMENT_1_RESULTS.zip'
print(f"📦 Packaging results into {zip_dest}...")
!zip -q -r {zip_dest} /kaggle/working/results/

if os.path.exists(zip_dest):
    print(f"✅ ZIP Archive ready: {zip_dest} ({os.path.getsize(zip_dest) / (1024**2):.2f} MB)")
    print("👉 Download the results directly from Kaggle output pane or click below:")
    display(FileLink('EXPERIMENT_1_RESULTS.zip'))
